# Freedom Superapp — Data Cleaning, EDA & Feature Engineering

**Datasets:**
- `users` — 1.1M rows — customer demographics
- `transactions` — 6M rows — bank card transactions
- `app_processes` — 9.5M rows — in-app user journeys
- `acquisition` — 1.1M rows — acquisition channel per customer
- `partner_purchases` — 215K rows — purchases inside Superapp partner services

## 0. Imports & settings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Consistent plot style throughout the notebook
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14

DATA_DIR = '/Users/adeliya/Desktop/freedom-hackaton-git/SAPP hackathon/'

## 1. Load data

In [ ]:
users            = pd.read_csv(DATA_DIR + 'SAPP пользователи.csv')
transactions     = pd.read_csv(DATA_DIR + 'SAPP транзакции.csv')
app_processes    = pd.read_csv(DATA_DIR + 'SAPP Процессы пользователей в приложении.csv')
acquisition      = pd.read_csv(DATA_DIR + 'SAPP привлечения.csv')
partner_purchases= pd.read_csv(DATA_DIR + 'SAPP Покупки у партнеров.csv')

print('Shapes:')
for name, df in [('users', users), ('transactions', transactions),
                 ('app_processes', app_processes), ('acquisition', acquisition),
                 ('partner_purchases', partner_purchases)]:
    print(f'  {name:20s}: {df.shape}')

## 2. Data Cleaning

### 2.1 Users

In [ ]:
print('=== USERS — raw info ===')
print(users.dtypes, '\n')
print('Nulls:\n', users.isnull().sum(), '\n')
print('Duplicates:', users.duplicated().sum())
users.head(3)

In [ ]:
# customer_id is float because of CSV encoding — cast to int (no nulls)
users['customer_id'] = users['customer_id'].astype(int)

# Parse registration date; errors='coerce' turns unparseable strings into NaT
users['reg_date'] = pd.to_datetime(users['reg_date'], errors='coerce')

# Sanity-check age: negative or unrealistically large values are noise
age_before = users['customer_age'].describe()
users = users[users['customer_age'].between(14, 100)]
print('Age range after filter:', users['customer_age'].min(), '-', users['customer_age'].max())

# Standardise gender to uppercase (defensive — data looks clean already)
users['gender'] = users['gender'].str.upper().str.strip()

print('\nUsers cleaned shape:', users.shape)

### 2.2 Transactions

In [ ]:
print('=== TRANSACTIONS — raw info ===')
print(transactions.dtypes, '\n')
print('Nulls:\n', transactions.isnull().sum(), '\n')
print('Duplicates:', transactions.duplicated(subset='transaction_id').sum())
transactions.head(3)

In [ ]:
# Cast ID columns
transactions['customer_id'] = transactions['customer_id'].astype(int)
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'], errors='coerce')

# MCC codes are categorical identifiers, not numeric values — keep as string
transactions['mcc'] = transactions['mcc'].astype(str).str.strip()

# transaction_sum: ~577K nulls (~9.5% of 6M rows).
# Strategy: keep rows but flag them — dropping would bias aggregations.
transactions['sum_is_missing'] = transactions['transaction_sum'].isna().astype(int)

# terminal_type: ~634K nulls — fill with explicit sentinel 'UNKNOWN'
transactions['terminal_type'] = transactions['terminal_type'].fillna('UNKNOWN')

# Drop exact duplicates on transaction_id (same payment recorded twice)
transactions = transactions.drop_duplicates(subset='transaction_id')

# Separate successful transactions for downstream analysis
txn_success = transactions[transactions['transaction_status'] == 'Успешные'].copy()

print('Transactions cleaned shape :', transactions.shape)
print('Successful transactions     :', txn_success.shape)

### 2.3 App Processes

In [ ]:
print('=== APP PROCESSES — raw info ===')
print(app_processes.dtypes, '\n')
print('Nulls:\n', app_processes.isnull().sum(), '\n')
app_processes.head(3)

In [ ]:
app_processes['customer_id'] = app_processes['customer_id'].astype(int)
app_processes['started_at']  = pd.to_datetime(app_processes['started_at'],  errors='coerce')
app_processes['completed_at']= pd.to_datetime(app_processes['completed_at'], errors='coerce')

# lang null (~13K rows, ~0.1%) — fill with 'UNKNOWN' rather than dropping;
# language doesn't affect process logic, just context
app_processes['lang'] = app_processes['lang'].fillna('UNKNOWN')

# completed_at null (327K) is structurally meaningful: process never finished
# We already capture this via status (ERROR / DECLINED), so no imputation needed
# Compute duration only where both timestamps exist
app_processes['duration_sec'] = (
    (app_processes['completed_at'] - app_processes['started_at'])
    .dt.total_seconds()
)

# Guard: negative durations = data entry error — set to NaN
app_processes.loc[app_processes['duration_sec'] < 0, 'duration_sec'] = np.nan

print('App processes cleaned shape:', app_processes.shape)

### 2.4 Acquisition

In [ ]:
print('=== ACQUISITION — raw info ===')
print(acquisition.dtypes, '\n')
print('Nulls:\n', acquisition.isnull().sum(), '\n')
acquisition.head(3)

In [ ]:
# 113 null customer_ids — cannot join, drop these rows
acquisition = acquisition.dropna(subset=['customer_id'])
acquisition['customer_id'] = acquisition['customer_id'].astype(int)

# 188K nulls in secondary_category_filled (~17%) — label them 'organic' / unknown
# rather than dropping: missing channel often means direct / organic acquisition
acquisition['secondary_category_filled'] = (
    acquisition['secondary_category_filled'].fillna('organic_unknown').str.strip().str.lower()
)

# Some customers may appear multiple times (multiple attribution touches)
print('Total acquisition rows :', len(acquisition))
print('Unique customers       :', acquisition['customer_id'].nunique())

# Keep the most-recent / first touch per customer (one row per customer for joins)
# Here we keep the first record per customer (data is not timestamped)
acquisition_dedup = acquisition.drop_duplicates(subset='customer_id', keep='first')
print('After dedup (1 row/customer):', acquisition_dedup.shape)

### 2.5 Partner Purchases

In [ ]:
print('=== PARTNER PURCHASES — raw info ===')
print(partner_purchases.dtypes, '\n')
print('Nulls:\n', partner_purchases.isnull().sum(), '\n')
partner_purchases.head(3)

In [ ]:
partner_purchases['customer_id']   = partner_purchases['customer_id'].astype(int)
partner_purchases['purchase_date'] = pd.to_datetime(partner_purchases['purchase_date'], errors='coerce')

# purchase_amount and cashback_amount are stored as negatives (debit convention)
# Flip signs so 'amount' means money spent (positive = spent more)
partner_purchases['purchase_amount']  = partner_purchases['purchase_amount'].abs()
partner_purchases['cashback_amount']  = partner_purchases['cashback_amount'].abs()

# Standardise app_name to lowercase for grouping
partner_purchases['app_name'] = partner_purchases['app_name'].str.strip().str.lower()

print('Partner purchases cleaned shape:', partner_purchases.shape)
print('Unique apps:', partner_purchases['app_name'].nunique())
print(partner_purchases['app_name'].value_counts().head(10))

## 3. Exploratory Data Analysis (EDA)

### 3.1 User Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Age distribution ---
# Understand the core customer segment; heavy right skew would suggest
# elderly outliers still in data
axes[0].hist(users['customer_age'], bins=40, edgecolor='white', color='steelblue')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# --- Gender split ---
gender_counts = users['gender'].value_counts()
axes[1].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#5B9BD5', '#ED7D31'])
axes[1].set_title('Gender Split')

# --- Top 15 cities ---
top_cities = users['city'].value_counts().head(15)
top_cities.plot(kind='barh', ax=axes[2], color='steelblue')
axes[2].set_title('Top 15 Cities by User Count')
axes[2].set_xlabel('Users')
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Registration trend — shows when user acquisition spikes occurred
reg_monthly = users.set_index('reg_date').resample('ME')['customer_id'].count()

fig, ax = plt.subplots(figsize=(14, 4))
reg_monthly.plot(ax=ax, marker='o', linewidth=1.5, color='steelblue')
ax.set_title('Monthly New User Registrations')
ax.set_xlabel('Month')
ax.set_ylabel('New Users')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 3.2 Transactions

In [ ]:
# Status breakdown — what fraction of attempted transactions succeed?
status_counts = transactions['transaction_status'].value_counts()
print('Transaction statuses:\n', status_counts, '\n')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Status bar chart
status_counts.head(10).plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Transaction Status Frequency')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

# Operation type breakdown
op_counts = transactions['operation_type'].value_counts().head(10)
op_counts.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Top 10 Operation Types')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Transaction amount distribution — only on successful rows with non-null amounts
# Use absolute value (amounts are signed debits)
amounts = txn_success['transaction_sum'].dropna().abs()

# Cap at 99th percentile to suppress extreme outliers in the chart
p99 = amounts.quantile(0.99)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(amounts[amounts <= p99], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Transaction Amount (≤ p99)')
axes[0].set_xlabel('Amount (KZT)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Log-scale view reveals the full distribution including large transactions
axes[1].hist(np.log1p(amounts), bins=60, color='coral', edgecolor='white')
axes[1].set_title('Transaction Amount (log1p scale)')
axes[1].set_xlabel('log1p(Amount)')

plt.tight_layout()
plt.show()

print(amounts.describe().apply(lambda x: f'{x:,.2f}'))

In [ ]:
# Top MCC codes by transaction volume — reveals what customers spend most on
top_mcc = (
    txn_success.groupby('mcc')['transaction_sum']
    .apply(lambda x: x.dropna().abs().sum())
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 5))
top_mcc.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 15 MCC Codes by Total Transaction Volume')
ax.set_xlabel('MCC Code')
ax.set_ylabel('Total Amount (KZT)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))
plt.tight_layout()
plt.show()

In [ ]:
# Daily transaction count trend — detect seasonality and data gaps
daily_txn = (
    txn_success.set_index('transaction_date')
    .resample('D')['transaction_id']
    .count()
)

fig, ax = plt.subplots(figsize=(14, 4))
daily_txn.plot(ax=ax, linewidth=0.8, color='steelblue')
ax.set_title('Daily Successful Transaction Count')
ax.set_ylabel('Transactions')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 3.3 App Processes

In [ ]:
# Process completion rate — core onboarding funnel health metric
status_pct = (
    app_processes.groupby(['process_code', 'status'])
    .size()
    .unstack(fill_value=0)
)
# Normalise to percentage per process
status_pct_norm = status_pct.div(status_pct.sum(axis=1), axis=0) * 100

status_pct_norm.plot(kind='bar', stacked=True, figsize=(14, 5),
                     colormap='tab10', edgecolor='white')
plt.title('Status Breakdown by Process (% of starts)')
plt.xlabel('Process')
plt.ylabel('%')
plt.xticks(rotation=30, ha='right')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Process duration distribution — outliers reveal UX or backend issues
dur = app_processes.dropna(subset=['duration_sec'])

# Cap at 95th pct per process to reduce chart distortion
fig, ax = plt.subplots(figsize=(14, 5))
for proc, grp in dur.groupby('process_code'):
    p95 = grp['duration_sec'].quantile(0.95)
    ax.hist(grp.loc[grp['duration_sec'] <= p95, 'duration_sec'],
            bins=50, alpha=0.5, label=proc)

ax.set_title('Process Duration Distribution (≤ p95 per process)')
ax.set_xlabel('Seconds')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

### 3.4 Partner Purchases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Revenue per app — shows which partner drives most GMV
app_revenue = (
    partner_purchases.groupby('app_name')['purchase_amount']
    .sum()
    .sort_values(ascending=False)
)
app_revenue.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Total Purchase Amount by App')
axes[0].set_ylabel('KZT')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))
axes[0].tick_params(axis='x', rotation=30)

# Cashback rate by app — which apps are most generous?
partner_purchases['cashback_rate'] = (
    partner_purchases['cashback_amount'] / partner_purchases['purchase_amount'].replace(0, np.nan)
)
cashback_by_app = partner_purchases.groupby('app_name')['cashback_rate'].median()
cashback_by_app.sort_values(ascending=False).plot(kind='bar', ax=axes[1],
                                                   color='coral', edgecolor='white')
axes[1].set_title('Median Cashback Rate by App')
axes[1].set_ylabel('Cashback / Purchase')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### 3.5 Acquisition Channels

In [ ]:
channel_counts = acquisition_dedup['secondary_category_filled'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 5))
channel_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 Acquisition Channels')
ax.set_xlabel('Users')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Feature Engineering

Goal: create one **customer-level feature table** suitable for segmentation or churn/LTV modelling.

### 4.1 Transaction features (per customer)

In [ ]:
# Reference date for recency calculation — latest date in the dataset
ref_date = txn_success['transaction_date'].max()
print('Reference date:', ref_date)

txn_feats = (
    txn_success
    .assign(abs_sum=lambda d: d['transaction_sum'].abs())
    .groupby('customer_id')
    .agg(
        txn_count            = ('transaction_id', 'count'),          # total successful transactions
        txn_total_spend      = ('abs_sum', 'sum'),                   # total money spent
        txn_avg_spend        = ('abs_sum', 'mean'),                  # average ticket size
        txn_median_spend     = ('abs_sum', 'median'),                # median — less sensitive to outliers
        txn_std_spend        = ('abs_sum', 'std'),                   # spend volatility
        txn_max_spend        = ('abs_sum', 'max'),                   # largest single transaction
        txn_unique_mcc       = ('mcc', 'nunique'),                   # breadth of merchant categories
        txn_unique_terminals = ('terminal_type', 'nunique'),         # channel diversity (POS/ePOS/ATM...)
        txn_first_date       = ('transaction_date', 'min'),
        txn_last_date        = ('transaction_date', 'max'),
    )
    .reset_index()
)

# Recency: days since last transaction — smaller = more recently active
txn_feats['txn_recency_days'] = (ref_date - txn_feats['txn_last_date']).dt.days

# Tenure in dataset: days between first and last transaction
txn_feats['txn_tenure_days'] = (
    (txn_feats['txn_last_date'] - txn_feats['txn_first_date']).dt.days
)

# Transaction frequency: avg transactions per active day
# Guard against divide-by-zero when tenure = 0 (single-day users)
txn_feats['txn_freq_per_day'] = (
    txn_feats['txn_count'] /
    txn_feats['txn_tenure_days'].replace(0, 1)
)

# Drop helper date columns — not needed as model features
txn_feats = txn_feats.drop(columns=['txn_first_date', 'txn_last_date'])

print('Transaction features shape:', txn_feats.shape)
txn_feats.head(3)

### 4.2 Operation-type features (pivot)

In [ ]:
# Each operation type gets its own share-of-wallet column.
# This captures behavioural mix (P2P, Purchase, ATM withdrawal, etc.)
op_pivot = (
    txn_success
    .groupby(['customer_id', 'operation_type'])
    .size()
    .unstack(fill_value=0)
)

# Normalise row-wise: proportion of each operation type
op_pivot_norm = op_pivot.div(op_pivot.sum(axis=1), axis=0)
op_pivot_norm.columns = ['op_share_' + c.replace(' ', '_').lower()
                          for c in op_pivot_norm.columns]
op_pivot_norm = op_pivot_norm.reset_index()

print('Op-type features shape:', op_pivot_norm.shape)
print('Columns:', list(op_pivot_norm.columns))

### 4.3 App Process features (per customer)

In [ ]:
# Completion rate: did the customer successfully finish their onboarding steps?
# Low rate → user may be frustrated or failing KYC
proc_feats = (
    app_processes
    .assign(is_completed=lambda d: (d['status'] == 'COMPLETED').astype(int))
    .groupby('customer_id')
    .agg(
        proc_total_attempts    = ('process_code', 'count'),
        proc_completed_count   = ('is_completed', 'sum'),
        proc_unique_processes  = ('process_code', 'nunique'),
        proc_avg_duration_sec  = ('duration_sec', 'mean'),
        proc_langs_used        = ('lang', 'nunique'),       # switching language = uncertainty signal
    )
    .reset_index()
)

# Completion rate — key quality signal for onboarding funnel
proc_feats['proc_completion_rate'] = (
    proc_feats['proc_completed_count'] / proc_feats['proc_total_attempts']
)

print('Process features shape:', proc_feats.shape)
proc_feats.head(3)

### 4.4 Partner Purchase features (per customer)

In [ ]:
pp_feats = (
    partner_purchases
    .groupby('customer_id')
    .agg(
        pp_purchase_count       = ('counter', 'sum'),          # total items bought
        pp_total_spend          = ('purchase_amount', 'sum'),
        pp_avg_spend            = ('purchase_amount', 'mean'),
        pp_total_cashback       = ('cashback_amount', 'sum'),
        pp_unique_apps          = ('app_name', 'nunique'),     # breadth of Superapp usage
        pp_avg_cashback_rate    = ('cashback_rate', 'mean'),
        pp_last_purchase_date   = ('purchase_date', 'max'),
    )
    .reset_index()
)

# Partner recency
pp_feats['pp_recency_days'] = (ref_date - pp_feats['pp_last_purchase_date']).dt.days
pp_feats = pp_feats.drop(columns='pp_last_purchase_date')

# Top app per customer — most-used partner service
top_app_per_user = (
    partner_purchases.groupby(['customer_id', 'app_name'])['counter']
    .sum()
    .reset_index()
    .sort_values('counter', ascending=False)
    .drop_duplicates(subset='customer_id')
    .rename(columns={'app_name': 'pp_top_app'})
    [['customer_id', 'pp_top_app']]
)
pp_feats = pp_feats.merge(top_app_per_user, on='customer_id', how='left')

print('Partner purchase features shape:', pp_feats.shape)
pp_feats.head(3)

### 4.5 Merge into one master customer table

In [ ]:
# Start with users — they are the unit of analysis
master = users.copy()

# Left join all feature tables: customers without activity get NaN
master = master.merge(acquisition_dedup[['customer_id', 'secondary_category_filled']]
                      .rename(columns={'secondary_category_filled': 'acq_channel'}),
                      on='customer_id', how='left')

master = master.merge(txn_feats,      on='customer_id', how='left')
master = master.merge(op_pivot_norm,  on='customer_id', how='left')
master = master.merge(proc_feats,     on='customer_id', how='left')
master = master.merge(pp_feats,       on='customer_id', how='left')

print('Master table shape:', master.shape)
print('\nNull counts (top 20 by nulls):')
print(master.isnull().sum().sort_values(ascending=False).head(20))

### 4.6 Derived / higher-order features

In [ ]:
# --- Superapp engagement score ---
# Composite: normalised count of active domains (transactions, partner apps, processes)
# Each component is clipped to [0, 1] before summing so one dimension can't dominate
master['engage_has_txn']    = (master['txn_count'].fillna(0) > 0).astype(int)
master['engage_has_pp']     = (master['pp_purchase_count'].fillna(0) > 0).astype(int)
master['engage_has_proc']   = (master['proc_total_attempts'].fillna(0) > 0).astype(int)
master['engagement_score']  = (
    master['engage_has_txn'] +
    master['engage_has_pp'] +
    master['engage_has_proc']
)  # 0 = dormant, 3 = fully active across all product areas

# --- Days since registration (account age) ---
master['account_age_days'] = (pd.Timestamp(ref_date) - master['reg_date']).dt.days

# --- RFM-style recency bucket ---
# Quintile bucketing: Q1 = most recent, Q5 = least recent
master['rfm_recency_bucket'] = pd.qcut(
    master['txn_recency_days'].fillna(master['txn_recency_days'].max() + 1),
    q=5, labels=[1, 2, 3, 4, 5], duplicates='drop'
)

# --- Spend frequency bucket ---
master['rfm_frequency_bucket'] = pd.qcut(
    master['txn_count'].fillna(0),
    q=5, labels=[1, 2, 3, 4, 5], duplicates='drop'
)

# --- Spend monetary bucket ---
master['rfm_monetary_bucket'] = pd.qcut(
    master['txn_total_spend'].fillna(0),
    q=5, labels=[1, 2, 3, 4, 5], duplicates='drop'
)

# --- Age group (for segmentation) ---
master['age_group'] = pd.cut(
    master['customer_age'],
    bins=[13, 24, 34, 44, 54, 100],
    labels=['18-24', '25-34', '35-44', '45-54', '55+']
)

# --- Partner adoption flag ---
# Whether customer has ever used any Superapp partner service
master['is_superapp_adopter'] = master['engage_has_pp']

print('Master table with derived features:', master.shape)
print('\nNew derived columns:')
new_cols = ['engagement_score', 'account_age_days', 'rfm_recency_bucket',
            'rfm_frequency_bucket', 'rfm_monetary_bucket', 'age_group', 'is_superapp_adopter']
print(master[new_cols].describe(include='all'))

### 4.7 Feature visualisation & validation

In [ ]:
# Engagement score distribution — what fraction of users are truly active?
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

eng_counts = master['engagement_score'].value_counts().sort_index()
eng_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Engagement Score Distribution\n(0=dormant, 3=fully active)')
axes[0].set_xlabel('Score')

# Spend vs. engagement
axes[1].boxplot(
    [master.loc[master['engagement_score'] == s, 'txn_total_spend'].dropna()
     for s in [0, 1, 2, 3]],
    labels=[0, 1, 2, 3],
    showfliers=False
)
axes[1].set_title('Total Spend by Engagement Score')
axes[1].set_xlabel('Engagement Score')
axes[1].set_ylabel('KZT')

# Superapp adoption by age group
adopt_by_age = (
    master.groupby('age_group', observed=True)['is_superapp_adopter']
    .mean() * 100
)
adopt_by_age.plot(kind='bar', ax=axes[2], color='coral', edgecolor='white')
axes[2].set_title('Superapp Adoption Rate by Age Group')
axes[2].set_ylabel('% Adopted')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap on key numeric features
# This helps detect multicollinearity before model training
corr_cols = [
    'customer_age', 'account_age_days',
    'txn_count', 'txn_total_spend', 'txn_avg_spend', 'txn_recency_days',
    'txn_unique_mcc', 'proc_completion_rate',
    'pp_purchase_count', 'pp_total_spend', 'pp_unique_apps',
    'engagement_score'
]

corr = master[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Export

In [ ]:
out_path = '/Users/adeliya/Desktop/freedom-hackaton-git/freedom-hackathon/master_features.csv'
master.to_csv(out_path, index=False)
print(f'Saved {master.shape[0]:,} rows × {master.shape[1]} columns → {out_path}')

print('\nFinal feature list:')
for col in master.columns:
    print(f'  {col}')